# IV-2 フーリエ解析と短時間フーリエ変換（ノートブック版）

教科書の IV-2 の主な演習を 1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。計算は全部で数秒〜数十秒です。音を聞いたり図を見比べたりする時間を含めると、20 分ほどが目安です）。

| 節 | 試すこと | 教科書 |
|---|---|---|
| 1 | DFT 行列の直交性と、自前の再帰 FFT と `numpy.fft` の照合 | IV-2-3-1・3-2、手順 A |
| 2 | 窓関数の性質とスペクトル漏れ | IV-2-3-3、手順 B |
| 3 | ゼロ詰め（分解能は上がらない）とうなり | IV-2-3-4、手順 B |
| 4 | STFT と逆 STFT：librosa・scipy との照合、完全再構成、COLA・NOLA・双対窓 | IV-2-3-5・3-6・3-10、手順 C |
| 5 | 窓の長さと、時間・周波数の分解能 | IV-2-3-7、手順 D |
| 6 | 位相：振幅だけ・位相だけで戻す、Griffin–Lim 法 | IV-2-3-8、手順 E |

- 次のセルたちが、共通ライブラリ（`common/stft.py`・`common/audio_io.py`・`common/plotting.py`）と `sounds.py` を、このノートブックの隣に書き出します。中身は教科書の 6 節と同じです。
- 各節の最後のセルが、教科書の「確かめること」と同じ値になるかを `assert` で確かめます。止まらずに「OK」と出れば合格です。
- **音を聞くときの注意**：再生ボタンは `listen()` が作ります。中で最大値を 0.3 にそろえ、`IPython.display.Audio(..., normalize=False)` で鳴らします。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前に、パソコンの音量を小さくしてください。イヤホンのときは特に注意します。
- 話し声は librosa の例の音源 `libri1` を使います（初回はダウンロードします）。ダウンロードできないときは、自動で合成の声に切り替わります（6 節の数値は少し変わります）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "librosa", librosa.__version__)
from scipy.signal import ShortTimeFFT   # scipy 1.12 より古いとここで ImportError（教科書 7 節の表）
import os
os.makedirs("common", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""


In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out


In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x


In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im


In [ ]:
%%writefile sounds.py
"""IV-2：演習で使う音（合成音と、librosa の例の音源・自分の録音）

  x = synth_vowel(fs, sec)   声に似た合成音（基本周波数 120→180 Hz、フォルマント 700・1200・2600 Hz）
  x = pluck(fs, sec, f0)     弦をはじいた音に似た合成音（Karplus–Strong 法）
  x, name = speech(fs, wav)  wav を指定すればそれ、無ければ librosa の例 libri1（ダウンロードが必要）、
                             それもだめなら synth_vowel
"""
import numpy as np
from scipy.signal import lfilter


def synth_vowel(fs=16000, sec=2.0, f0=(120.0, 180.0), formants=((700, 80), (1200, 90), (2600, 120))):
    """声帯の音（パルス列）を、声道の共振（2次の共振器を3つ直列）に通した「あ」に近い音"""
    n = int(sec * fs)
    f = np.linspace(f0[0], f0[1], n)                       # 基本周波数をゆっくり上げる
    phase = np.cumsum(f / fs)
    src = np.diff(np.floor(phase), prepend=0.0)            # 1周期に1回だけ 1 になるパルス列
    y = src
    for fc, bw in formants:                                # 中心 fc [Hz]・帯域幅 bw [Hz] の共振器
        r = np.exp(-np.pi * bw / fs)
        a = [1.0, -2 * r * np.cos(2 * np.pi * fc / fs), r * r]
        y = lfilter([1.0 - r], a, y)
    env = np.minimum(1.0, np.minimum(np.arange(n), n - np.arange(n)) / (0.05 * fs))   # 前後 50ms で立ち上げ・下げ
    y = y * env
    return y / np.max(np.abs(y)) * 0.5


def pluck(fs=16000, sec=2.0, f0=196.0, seed=0):
    """Karplus–Strong 法：雑音を1周期分の遅延線に入れ、平均をとりながら回す（弦の減衰に似る）"""
    rng = np.random.default_rng(seed)
    n = int(sec * fs)
    P = int(round(fs / f0))
    buf = rng.uniform(-1, 1, P)
    y = np.zeros(n)
    for i in range(n):
        y[i] = buf[i % P]
        buf[i % P] = 0.996 * 0.5 * (buf[i % P] + buf[(i + 1) % P])
    return y / np.max(np.abs(y)) * 0.5


def speech(fs=16000, wav=None, sec=None):
    """話し声を返す。(x, 名前)"""
    from common.audio_io import load
    if wav:
        x, _ = load(wav, fs)
        if x.ndim == 2:
            x = x[:, 0]
        name = "own"
    else:
        try:
            import librosa
            x, _ = librosa.load(librosa.ex("libri1"), sr=fs, mono=True)
            name = "libri1"
        except Exception as e:                              # ダウンロードできないときなど
            print(f"librosa の例の音源が使えないので合成音にします（{type(e).__name__}）")
            x, name = synth_vowel(fs, 3.0), "synth_vowel"
    if sec is not None:
        x = x[:int(sec * fs)]
    return x / (np.max(np.abs(x)) + 1e-12) * 0.5, name


In [ ]:
import sys, time, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.signal import find_peaks
import librosa
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common.stft import stft, istft, get_window, dual_window, window_sum
from common.plotting import specshow, to_db       # import すると Hiragino Sans が設定される
from sounds import speech, pluck
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="Glyph")

FS = 16000
EPS = np.finfo(float).tiny


def listen(x, fs=FS):
    """最大 0.3 にそろえて再生ボタンを出す。normalize=False を消さない"""
    x = np.asarray(x, dtype=float)
    display(Audio(x / (np.max(np.abs(x)) + 1e-12) * 0.3, rate=fs, normalize=False))


def rel(a, b):
    """相対誤差の最大 max|a − b| / max|b|"""
    return np.max(np.abs(a - b)) / (np.max(np.abs(b)) + 1e-300)


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)

## 1. DFT 行列と FFT（教科書 IV-2-3-1・3-2、手順 A）

DFT 行列 $F[k,n] = e^{-j2\pi kn/N}$ の列は直交していて、$\mathbf F^{\mathsf H}\mathbf F = N\mathbf I$ です。
再帰の FFT は $X[k] = E[k] + W^kO[k]$、$X[k+N/2] = E[k] - W^kO[k]$（$E$ は偶数番目、$O$ は奇数番目の DFT）をくり返します。

In [ ]:
def dft_matrix(N):
    """DFT 行列 F[k, n] = exp(−j2πkn/N)。X = F x"""
    k = np.arange(N)
    return np.exp(-2j * np.pi * np.outer(k, k) / N)


def fft_rec(x):
    """基数2・時間間引きの FFT（再帰）。N は2のべき乗"""
    x = np.asarray(x, dtype=complex)
    N = x.shape[0]
    if N == 1:
        return x.copy()
    if N % 2:
        raise ValueError("長さは2のべき乗にしてください")
    E = fft_rec(x[0::2])
    O = fft_rec(x[1::2])
    tw = np.exp(-2j * np.pi * np.arange(N // 2) / N) * O     # 回転因子 W^k を掛ける
    return np.concatenate([E + tw, E - tw])                   # X[k] = E + W^k O、X[k+N/2] = E − W^k O


def ifft_rec(X):
    """逆 FFT：x = (1/N) conj(FFT(conj(X)))"""
    X = np.asarray(X, dtype=complex)
    return np.conj(fft_rec(np.conj(X))) / X.shape[0]


rng = np.random.default_rng(0)
F4 = dft_matrix(4)
print("F4 の実部\n", np.round(F4.real).astype(int), "\nF4 の虚部\n", np.round(F4.imag).astype(int))
x4 = np.array([1.0, 2.0, 0.0, -1.0])
X4 = F4 @ x4
print("x =", x4, "→ F x =", np.round(X4, 12))
print("手で：E = DFT([1, 0]) =", fft_rec([1, 0]), "、O = DFT([2, −1]) =", fft_rec([2, -1]))

orth = {}
for N in (4, 8, 64):
    F = dft_matrix(N)
    orth[N] = np.max(np.abs(F.conj().T @ F - N * np.eye(N)))
    print(f"N = {N:3d}: max|F^H F − N I| = {orth[N]:.1e}")
F = dft_matrix(64)
x = rng.standard_normal(64)
X = F @ x
print(f"逆変換の誤差 {np.max(np.abs(x - F.conj().T @ X / 64)):.1e}、パーセバル Σ|x|² = {np.sum(x**2):.6f}、Σ|X|²/N = {np.sum(np.abs(X)**2)/64:.6f}")

fft_err = {}
for N in (2, 8, 64, 1024, 4096, 65536):
    x = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    Xn = np.fft.fft(x)
    fft_err[N] = (rel(fft_rec(x), Xn), rel(ifft_rec(Xn), x))
    print(f"N = {N:6d}: 自前の FFT と numpy の相対誤差 {fft_err[N][0]:.1e}、逆 FFT {fft_err[N][1]:.1e}")
for N in (8, 1024, 65536):
    print(f"N = {N:6d}: 掛け算の数 DFT {N*N}、FFT {N//2*int(np.log2(N))}、比 {N*N/(N//2*int(np.log2(N))):.1f}")
x = rng.standard_normal(4096) + 0j
F = dft_matrix(4096)
t0 = time.perf_counter(); F @ x; t1 = time.perf_counter(); fft_rec(x); t2 = time.perf_counter(); np.fft.fft(x); t3 = time.perf_counter()
print(f"N = 4096 の時間：行列 {1e3*(t1-t0):.1f} ms、再帰 FFT {1e3*(t2-t1):.1f} ms、numpy {1e3*(t3-t2):.3f} ms（計算機で大きく違う）")

In [ ]:
# 確かめ（手順 A）
ok(np.allclose(X4, [2, 1 - 3j, 0, 1 + 3j], atol=1e-12), "x = [1, 2, 0, −1] の DFT が [2, 1−3j, 0, 1+3j]")
ok(np.allclose(fft_rec([1, 0]), [1, 1]) and np.allclose(fft_rec([2, -1]), [1, 3]), "E = [1, 1]、O = [1, 3]（手の計算と同じ）")
ok(max(orth.values()) < 1e-12, f"max|F^H F − N I| が 1e-12 より小さい（丸め誤差の大きさ。N = 64 で最大 {max(orth.values()):.1e}）")
ok(max(e for pair in fft_err.values() for e in pair) < 1e-14, "再帰 FFT と numpy の相対誤差が N = 65536 まで 1e-15 の桁")
ok(abs(1024 * 1024 / (512 * 10) - 204.8) < 1e-9, "掛け算の比が N = 1024 で 204.8")

## 2. 窓関数とスペクトル漏れ（教科書 IV-2-3-3、手順 B）

単位は「ビン」（$f_s/L$）です。ENBW（等価雑音帯域幅）、−3 dB 幅、最大のサイドローブ、スカラップ損失（周波数がビンの真ん中にあるときの低下）を、窓のスペクトルを 64 倍に細かくして数値で求めます。

In [ ]:
WINDOWS = ["boxcar", "hann", "hamming", "blackman", ("kaiser", 8.0)]
L, OVER = 1024, 64


def wname(w):
    return w if isinstance(w, str) else f"{w[0]}({w[1]:g})"


def window_props(win, L=L, over=OVER):
    w = get_window(win, L)
    W = np.abs(np.fft.rfft(w, L * over))
    W /= W[0]
    db = 20 * np.log10(W + 1e-300)
    b = np.arange(len(W)) / over                   # 単位：ビン

    def cross(level):                              # db が level を下回る点を直線補間で求める
        i = np.argmax(db < level)
        return b[i - 1] + (db[i - 1] - level) / (db[i - 1] - db[i]) * (b[i] - b[i - 1])
    inull = np.argmax(np.diff(W) > 0)              # 主ローブの終わり（最初の極小）
    return dict(window=wname(win), coherent_gain=w.sum() / L, enbw_bins=L * np.sum(w ** 2) / w.sum() ** 2,
                bw3_bins=2 * cross(-3.0103), bw6_bins=2 * cross(-6.0206), mainlobe_bins=2 * b[inull],
                sidelobe_db=db[inull:].max(), scallop_db=db[over // 2])


def tone(f, fs, n, a=1.0):
    return a * np.sin(2 * np.pi * f * np.arange(n) / fs)


tab = pd.DataFrame([window_props(w) for w in WINDOWS])
display(tab.round(3))

print("スペクトル漏れ：振幅 1 の正弦波（ビンの上 k = 64 と、ビンの真ん中 k = 64.5）")
amp_read = {}
for f_bin in (64.0, 64.5):
    for win in ("boxcar", "hann"):
        w = get_window(win, L)
        amp = 2 * np.abs(np.fft.rfft(w * tone(f_bin * FS / L, FS, L))) / w.sum()   # コヒーレント利得で割って振幅に
        far = 20 * np.log10(amp[64 + 20] + 1e-300)
        amp_read[(f_bin, win)] = amp.max()
        print(f"  k = {f_bin:4.1f} {win:7s}: 振幅の読み {amp.max():.3f}、20 ビン離れたところ "
              + ("漏れなし（丸め誤差の大きさ）" if far < -200 else f"{far:6.1f} dB"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for win in WINDOWS:
    w = get_window(win, L)
    W = np.abs(np.fft.rfft(w, L * OVER))
    axes[0].plot(np.arange(len(W)) / OVER, to_db(W), lw=1, label=wname(win))
    x = tone(64.5 * FS / L, FS, L) + tone(80.0 * FS / L, FS, L, 10 ** (-50 / 20))
    axes[1].plot(np.arange(L // 2 + 1), to_db(np.fft.rfft(w * x)), lw=1, label=wname(win))
axes[0].set(xlim=(0, 12), ylim=(-120, 3), xlabel="周波数のずれ [ビン]", ylabel="dB（最大を 0）", title="窓のスペクトル")
axes[1].set(xlim=(40, 110), ylim=(-140, 3), xlabel="ビン k", ylabel="dB（最大を 0）", title="強い音（k=64.5）と 50dB 小さい音（k=80）")
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
# 確かめ（手順 B の 4）：3-3 節の表と同じ値
h = tab.set_index("window").loc["hann"]
ok(round(h.enbw_bins, 3) == 1.500, f"hann の ENBW 1.500（{h.enbw_bins:.3f}）")
ok(round(h.bw3_bins, 3) == 1.441, f"hann の −3 dB 幅 1.441（{h.bw3_bins:.3f}）")
ok(round(h.sidelobe_db, 1) == -31.5, f"hann の最大のサイドローブ −31.5 dB（{h.sidelobe_db:.1f}）")
ok(round(h.scallop_db, 2) == -1.42, f"hann のスカラップ損失 −1.42 dB（{h.scallop_db:.2f}）")
ok(round(amp_read[(64.5, "boxcar")], 3) == 0.639 and round(amp_read[(64.5, "hann")], 3) == 0.849,
   "ビンの真ん中の正弦波の振幅の読み：方形窓 0.639、hann 0.849")

## 3. ゼロ詰め（教科書 IV-2-3-4、手順 B）

1000 Hz と 1010 Hz（差 10 Hz）の2つの正弦波を、窓 1024 点（64 ms）と 4096 点（256 ms）で見ます。FFT の長さを 4 倍・16 倍にしても（ゼロ詰め）、山の数は変わりません。分けられるかどうかは**窓の長さ**で決まります。

In [ ]:
peaks = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, Lw in zip(axes, (1024, 4096)):
    x = tone(1000, FS, Lw) + tone(1010, FS, Lw)
    w = get_window("hann", Lw)
    for nfft in (Lw, 4 * Lw, 16 * Lw):
        X = np.abs(np.fft.rfft(w * x, nfft))
        fr = np.arange(len(X)) * FS / nfft
        band = (fr > 950) & (fr < 1060)
        pk, _ = find_peaks(X[band], height=0.5 * X[band].max())   # 最大の半分（−6 dB）より高い山だけ数える
        peaks[(Lw, nfft)] = len(pk)
        print(f"窓 L = {Lw:5d}（{Lw/FS*1000:4.0f} ms）、FFT {nfft:6d} 点（点の間隔 {FS/nfft:5.2f} Hz）：大きな山の数 {len(pk)}")
        if nfft != 4 * Lw:
            ax.plot(fr, X / X.max(), "o-" if nfft == Lw else "-", ms=4, lw=1, label=f"FFT {nfft} 点")
    ax.axvline(1000, color="gray", ls=":"); ax.axvline(1010, color="gray", ls=":")
    ax.set(xlim=(950, 1060), xlabel="周波数 [Hz]", title=f"窓 {Lw} 点（{Lw/FS*1000:.0f} ms）")
    ax.legend(); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("振幅（最大を 1）")
fig.tight_layout(); plt.show()

# 耳で：2つの正弦波は 1 秒に 10 回のうなり。1005 Hz の1つの音はゆれない（音量を小さくしてから）
n = 2 * FS
ramp = np.minimum(1, np.minimum(np.arange(n), n - np.arange(n)) / (0.02 * FS))
print("1000 Hz + 1010 Hz（うなり）")
listen((tone(1000, FS, n) + tone(1010, FS, n)) * ramp)
print("1005 Hz だけ")
listen(tone(1005, FS, n) * ramp)

In [ ]:
# 確かめ（手順 B の 5）
ok(all(peaks[(1024, m)] == 1 for m in (1024, 4096, 16384)), "窓 1024 点（64 ms）ではゼロ詰めしても山が 1 つ（3 行とも）")
ok(all(peaks[(4096, m)] == 2 for m in (4096, 16384, 65536)), "窓 4096 点（256 ms）では山が 2 つ（3 行とも）")

## 4. STFT と逆 STFT（教科書 IV-2-3-5・3-6・3-10、手順 C）

共通ライブラリ `common.stft` を、librosa と `scipy.signal.ShortTimeFFT` と数値で照らし合わせ、完全再構成を確かめます。
scipy の既定は「窓の中心」が位相の基準なので、`phase="center"` にするとそろいます。

In [ ]:
rng = np.random.default_rng(0)
n = 16000 + 123
x = rng.standard_normal(n)

print("== librosa との照合（x は乱数 16123 点）")
lib_err = []
for L, S, win in ((1024, 256, "hann"), (2048, 512, "hann"), (512, 128, "hamming"),
                  (4096, 2048, "hamming"), (1024, 512, "sqrthann"), (1000, 250, "blackman")):
    w = get_window(win, L)
    X = stft(x, L, S, win)
    e1 = rel(X, librosa.stft(x, n_fft=L, hop_length=S, window=w))
    Y = X * rng.uniform(0, 1, X.shape)                       # 加工した STFT（乱数のマスク）
    e2 = rel(istft(Y, L, S, win, length=n), librosa.istft(Y, hop_length=S, n_fft=L, window=w, length=n))
    lib_err += [e1, e2]
    print(f"L={L:5d} S={S:5d} {win:9s} 形 {X.shape}: stft {e1:.1e}、加工後の istft {e2:.1e}")

print("\n== scipy.signal.ShortTimeFFT との照合（L = 1024、S = 256、hann）")
L, S = 1024, 256
w = get_window("hann", L)
SFT = signal.ShortTimeFFT(w, S, FS)
Xs = SFT.stft(x)
X = stft(x, L, S, "hann")
Xs0 = SFT.stft(x, p0=0, p1=X.shape[1])
sc = {}
sc["a"] = rel(X, Xs0)
sc["b"] = rel(stft(x, L, S, "hann", phase="center"), Xs0)
k = np.arange(L // 2 + 1)
sc["c"] = rel(X * ((-1.0) ** k)[:, None], Xs0)
sc["d"] = rel(X, signal.ShortTimeFFT(w, S, FS, phase_shift=None).stft(x, p0=0, p1=X.shape[1]))
pre, post = -SFT.p_min * S, (SFT.p_max(n) - 1 - n // S) * S
xpad = np.concatenate([np.zeros(pre), x, np.zeros(post)])
sc["e"] = rel(stft(xpad, L, S, "hann", phase="center"), Xs)
Y = Xs * rng.uniform(0, 1, Xs.shape)
sc["f"] = rel(istft(Y, L, S, "hann", length=len(xpad), phase="center")[pre:pre + n], SFT.istft(Y, k1=n))
sc["g"] = rel(dual_window(w, S), SFT.dual_win)
print(f"フレームの数：scipy {Xs.shape[1]} 個、common.stft {X.shape[1]} 個")
for key, s in zip("abcdefg", ["範囲だけそろえる（位相の基準が違う）", "phase='center'", "差は (−1)^k だけ",
                             "scipy で phase_shift=None", f"前に {pre}・後ろに {post} 個のゼロ → 全フレーム",
                             "加工した STFT の逆変換", "双対窓 dual_win"]):
    print(f"({key}) {s}: {sc[key]:.1e}")

print("\n== 完全再構成（stft → istft）")
recon = {}
for L, S, win in ((1024, 256, "hann"), (1024, 384, "hann"), (1024, 512, "blackman"),
                  (4096, 2048, "hamming"), (1024, 512, "sqrthann")):
    recon[(L, S, win)] = np.max(np.abs(istft(stft(x, L, S, win), L, S, win, length=n) - x))
    print(f"L={L} S={S} {win:9s}: 最大誤差 {recon[(L, S, win)]:.1e}")
ws = np.hanning(1024)                                           # 左右対称な hann（両端が 0）
e_sym = np.max(np.abs(istft(stft(x, 1024, 256, ws), 1024, 256, ws, length=n) - x))
e_mis = np.max(np.abs(istft(stft(x, 1024, 256, ws), 1024, 256, "hann", length=n) - x))
print(f"numpy.hanning を両方に：{e_sym:.1e}、stft は numpy.hanning・istft は周期的な hann（食い違い）：{e_mis:.1e}")

x32 = x.astype(np.float32)
e7 = rel(stft(x32, 2048, 512), librosa.stft(x32, n_fft=2048, hop_length=512))
print(f"\n第7章（librosa、float32 → complex64）との相対誤差 {e7:.1e}（単精度の丸め誤差の大きさ）")

In [ ]:
print("COLA と NOLA（L = 1024）。Σ_t w[n−tS] の 最小/最大 と、Σ_t w²[n−tS] の最小（端を除いた真ん中）")
rows = []
L = 1024
for win in ("boxcar", "hann", "hamming", "sqrthann", "blackman"):
    w = get_window(win, L)
    for S in (1024, 512, 384, 256):
        T = 40
        s1 = np.zeros((T - 1) * S + L)
        for t in range(T):
            s1[t * S:t * S + L] += w
        s2 = window_sum(w, S, T)
        mid = slice(L, (T - 1) * S)
        r = s1[mid].min() / s1[mid].max()
        rows.append(dict(窓=win, S=S, Σw最小最大=round(r, 4), COLA=bool(r > 1 - 1e-10),
                         COLA_scipy=bool(signal.check_COLA(w, L, L - S)), Σw2最小=round(s2[mid].min(), 4),
                         NOLA=bool(s2[mid].min() > 1e-10 * s2.max()), NOLA_scipy=bool(signal.check_NOLA(w, L, L - S))))
cola = pd.DataFrame(rows)
display(cola)

w = get_window("hann", 1024)
g = dual_window(w, 256)
print("hann、S = L/4 の双対窓：g = w / {:.4f}".format(w[512] / g[512]))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(w, label="分析窓 w（hann）"); ax.plot(g, label="双対窓 g = w / Σw²"); ax.plot(window_sum(w, 256, 8)[:2048], label="Σw²（8 フレーム）")
ax.set(xlabel="n", title="hann、L = 1024、S = 256"); ax.legend(); ax.grid(True, alpha=0.3); plt.show()

print("hann、S = L（重なりなし）は NOLA が成り立たない。istft はエラーを出すはず：")
nola_error = False
try:
    istft(stft(x, 1024, 1024, "hann"), 1024, 1024, "hann", length=n)
except ValueError as e:
    nola_error = True
    print("  ValueError:", e)

In [ ]:
# 確かめ（手順 C の 7）
ok(max(lib_err) < 1e-10, f"librosa と 6 行すべて一致（最大 {max(lib_err):.1e}）")
ok(sc["a"] > 1.0, f"(a) だけ不一致（{sc['a']:.2f}。教科書では約 1.9）")
ok(max(sc[k] for k in "bcdefg") < 1e-10, "(b)〜(g) は一致")
ok((Xs.shape[1], X.shape[1]) == (66, 63), "フレームの数が scipy 66 個・common.stft 63 個")
ok(max(recon.values()) < 1e-13 and e_sym < 1e-13, "完全再構成の最大誤差が 1e-15 の桁（numpy.hanning を両方に渡しても同じ）")
ok(1e-4 < e_mis < 1e-2, f"窓を食い違えた行だけ 1e-3 の桁（{e_mis:.1e}）")
hn = cola[cola["窓"] == "hann"].set_index("S")
ok(hn.loc[512, "COLA"] and hn.loc[256, "COLA"] and not hn.loc[384, "COLA"] and hn.loc[384, "NOLA"] and not hn.loc[1024, "NOLA"],
   "hann：S = 512・256 で COLA、S = 384 は COLA でないが NOLA、S = 1024 で NOLA でない")
ok((cola["COLA"] == cola["COLA_scipy"]).all() and (cola["NOLA"] == cola["NOLA_scipy"]).all(), "COLA・NOLA の判定が scipy と同じ列")
ok(nola_error, "S = L の hann で istft が ValueError（NOLA）を出す")
ok(1e-9 < e7 < 1e-6, f"第7章（float32）との相対誤差が 1e-8 の桁（{e7:.1e}）")

## 5. 窓の長さと、時間・周波数の分解能（教科書 IV-2-3-7、手順 D）

0.25 秒ごとのクリックと、1000 Hz・1030 Hz（差 30 Hz）の正弦波を、窓の長さ 4 通りで見ます。短い窓は時間に鋭く、長い窓は周波数に鋭くなります。
続けて話し声を、振幅だけ残して位相を乱数にして戻し、窓の長さで崩れ方がどう変わるかを聞きます。

In [ ]:
LS = (128, 512, 2048, 8192)                 # 16 kHz で 8・32・128・512 ms
nt = 2 * FS
t = np.arange(nt) / FS
xt = 0.2 * (np.sin(2 * np.pi * 1000 * t) + np.sin(2 * np.pi * 1030 * t))
xt[(np.arange(1, 8) * FS // 4)] += 1.0
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, L in zip(axes.ravel(), LS):
    S = L // 4
    specshow(stft(xt, L, S), FS, S, ax=ax, db_range=70, colorbar=False, title=f"L = {L}（{L/FS*1000:.0f} ms）、fs/L = {FS/L:.1f} Hz")
    ax.set_ylim(700, 1300)
fig.suptitle("クリック（0.25 秒ごと）と 1000 Hz・1030 Hz の正弦波"); fig.tight_layout(); plt.show()

xs, src_name = speech(FS, None, sec=4.0)
n = len(xs)
print(f"音源：{src_name}（{n/FS:.1f} 秒）。まず元の音")
listen(xs)
rng = np.random.default_rng(0)
tr_err = {}
for L in LS:
    S = L // 4
    X = stft(xs, L, S)
    tr_err[L] = np.max(np.abs(istft(X, L, S, length=n) - xs))
    A = np.abs(X)
    yr = istft(A * np.exp(1j * rng.uniform(-np.pi, np.pi, A.shape)), L, S, length=n)
    print(f"L = {L:5d}（{L/FS*1000:4.0f} ms）：そのまま戻した誤差 {tr_err[L]:.1e}。位相を乱数にした音 ↓")
    listen(yr)

In [ ]:
# 確かめ（手順 D の 9）と、聞いたことのメモ
ok(max(tr_err.values()) < 1e-14, "そのまま戻した誤差が 4 通りとも 1e-16 の桁")
print("窓が短いと「ざらざら・がさがさ」、長いと「にじむ・お風呂のような響き」に聞こえるか、ノートに書きましょう。")

## 6. 位相の役割と Griffin–Lim 法（教科書 IV-2-3-8、手順 E）

まず、位相が持っている情報（瞬時周波数・群遅延）を数値で見ます。次に、振幅だけ・位相だけで戻した音を聞きます。
最後に、振幅だけから位相を推定する Griffin–Lim 法を自分で書き、`librosa.griffinlim` と照らし合わせます。

In [ ]:
L1, S1 = 1024, 256
f0 = 1003.0                                              # ビン（15.625 Hz 刻み）の間にある周波数
X = stft(np.sin(2 * np.pi * f0 * np.arange(FS) / FS), L1, S1)
k = int(np.argmax(np.abs(X[:, 10])))
dphi = np.angle(X[k, 11]) - np.angle(X[k, 10])           # 隣のフレームとの位相の差
dev = np.angle(np.exp(1j * (dphi - 2 * np.pi * k * S1 / L1)))   # ビンの中心から期待される進みとのずれ
f_est = (k + dev * L1 / (2 * np.pi * S1)) * FS / L1
print(f"一番大きいビン k = {k}（{k*FS/L1:.3f} Hz）。位相の進みから求めた周波数 {f_est:.3f} Hz（本当は {f0} Hz）")
D = 37
hh = np.zeros(L1); hh[D] = 1.0
om = 2 * np.pi * np.arange(L1 // 2 + 1) / L1
gd = -np.gradient(np.unwrap(np.angle(np.fft.rfft(hh))), om)
print(f"{D} 点遅らせたインパルスの群遅延 −dφ/dω：平均 {gd.mean():.3f}、最小 {gd.min():.3f}、最大 {gd.max():.3f}")

In [ ]:
def wnorm(D):
    """片側スペクトル D（F×T）から両側スペクトル全体のノルム（直流とナイキスト以外は重み 2）"""
    wt = np.full(D.shape[0], 2.0)
    wt[0] = wt[-1] = 1.0
    return np.sqrt(np.sum(wt[:, None] * np.abs(D) ** 2))


L, S = 1024, 256
X = stft(xs, L, S)
A, ph = np.abs(X), np.angle(X)
G = stft(pluck(FS, n / FS, f0=110.0), L, S)
outs = {
    "zero（振幅だけ、位相 0）": istft(A, L, S, length=n),
    "random（振幅だけ、位相は乱数）": istft(A * np.exp(1j * rng.uniform(-np.pi, np.pi, A.shape)), L, S, length=n),
    "only（位相だけ、振幅は全部同じ）": istft(np.exp(1j * ph), L, S, length=n),
    "swapA（声の振幅 × 弦の位相）": istft(A * np.exp(1j * np.angle(G)), L, S, length=n),
    "swapB（弦の振幅 × 声の位相）": istft(np.abs(G) * np.exp(1j * ph), L, S, length=n),
}
for name, y in outs.items():
    sc_ = wnorm(np.abs(stft(y, L, S)) - A) / wnorm(A)
    print(f"{name}：‖|STFT(y)| − A‖ / ‖A‖ = {sc_:.3f}")
    listen(y)
print(f"位相 0 の音には、フレームの間隔 S/fs = {S/FS*1000:.0f} ms ごとのくり返し（{FS/S:.1f} Hz のブザー）が混じる")

In [ ]:
def griffin_lim(A, L, S, window="hann", n_iter=100, length=None, momentum=0.0):
    """振幅 A だけから信号を探す。x ← istft(A·e^{jφ})、φ ← ∠stft(x) をくり返す（Griffin & Lim 1984）。
    momentum > 0 は librosa.griffinlim と同じ形の加速。戻り値：x と毎回の誤差 ‖|stft(x)| − A‖ / ‖A‖"""
    C = A * np.ones_like(A, dtype=complex)                  # 位相 0 から始める（librosa の init=None と同じ）
    prev, hist = None, []
    for _ in range(n_iter):
        x = istft(C, L, S, window, length=length)
        R = stft(x, L, S, window)
        hist.append(wnorm(np.abs(R) - A) / wnorm(A))
        C = R.copy()
        if prev is not None and momentum > 0:
            C -= (momentum / (1 + momentum)) * prev
        C = A * C / (np.abs(C) + EPS)                        # 振幅を A に戻し、位相だけ受け取る
        prev = R
    return istft(C, L, S, window, length=length), np.array(hist)


gl_err = []
for it in (1, 10, 32):
    y_my, _ = griffin_lim(A, L, S, n_iter=it, length=n)
    y_lr = librosa.griffinlim(A, n_iter=it, hop_length=S, n_fft=L, window="hann", length=n, momentum=0.0, init=None)
    gl_err.append(rel(y_my, y_lr))
    print(f"{it:3d} 回：librosa.griffinlim との相対誤差の最大 {gl_err[-1]:.1e}")
y32, h32 = griffin_lim(A, L, S, n_iter=32, length=n)
y100, h100 = griffin_lim(A, L, S, n_iter=100, length=n)
yf, hf = griffin_lim(A, L, S, n_iter=100, length=n, momentum=0.99)
n_up = int(np.sum(np.diff(h100) > 1e-12))
print(f"誤差：1 回目 {h100[0]:.3f} → 32 回 {h32[-1]:.3f} → 100 回 {h100[-1]:.3f}、加速版 100 回 {hf[-1]:.3f}。誤差が増えた回数 {n_up}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(np.arange(1, 101), h100, label="Griffin–Lim"); ax.semilogy(np.arange(1, 101), hf, label="加速版（momentum 0.99）")
ax.set(xlabel="反復回数", ylabel="‖|STFT(x)| − A‖ / ‖A‖", title="Griffin–Lim 法の誤差の減り方")
ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()
for name, y in [("元の音", xs), ("GL 32 回", y32), ("GL 100 回", y100), ("加速版 100 回", yf)]:
    print(name); listen(y)

In [ ]:
# 確かめ（手順 E の 12）。6 節の後半の値は libri1 を使ったときのもの
ok(abs(f_est - 1003.0) < 5e-4, f"位相の進みから求めた周波数 1003.000 Hz（{f_est:.3f}）")
ok(abs(gd.min() - 37) < 1e-6 and abs(gd.max() - 37) < 1e-6, "群遅延が 37.000（最小・最大とも）")
ok(max(gl_err) <= 1e-11, f"自前の Griffin–Lim と librosa の相対誤差が 1e-11 以下（{max(gl_err):.1e}。librosa・numpy の版で桁が変わるので幅を持たせてある）")
ok(n_up == 0, "加速なしの Griffin–Lim で誤差が増えた回数 0（Griffin–Lim の定理）")
if src_name == "libri1":
    ok(abs(h100[0] - 0.893) < 0.01, f"誤差 1 回目 0.893（{h100[0]:.3f}）")
    ok(abs(h32[-1] - 0.12) < 0.015 and abs(h100[-1] - 0.064) < 0.01 and abs(hf[-1] - 0.03) < 0.01,
       f"32 回で約 0.12、100 回で約 0.06、加速版 100 回で約 0.03（{h32[-1]:.3f}、{h100[-1]:.3f}、{hf[-1]:.3f}）")
else:
    print("合成の声なので、誤差の値の確かめはとばしました")

## おわりに

- 自前の DFT・FFT・窓の表・STFT・Griffin–Lim が、numpy・scipy・librosa と一致することを確かめました。
- このノートブックに入れていないもの：スペクトログラムの練習帳（手順 F、`spectrogram_gallery.py`）、統計の誤用（手順 G、`periodogram_stats.py`）、確認問題の答え合わせ（`quiz_check.py`）。教科書の 6 節のスクリプトで試し、7 節の確認問題も解いてみてください。
- 自分の声で試すときは、録音した wav をこのノートブックの隣に置き、5 節の `speech(FS, None, sec=4.0)` の `None` を `"ファイル名.wav"` に変えます。